# Medical Diagnostics Multi-Agent System — Demo Walkthrough

This notebook walks through the full diagnostic pipeline step by step.
It's useful for debugging, understanding what each agent is doing,
and experimenting with different patient inputs.

**Prerequisites:** `ANTHROPIC_API_KEY` must be set in your environment.

```bash
export ANTHROPIC_API_KEY=sk-ant-...
```


In [ ]:
import os
import sys
import json

# make sure we're running from project root
if not os.path.exists('agents'):
    os.chdir('..')
    
assert os.getenv('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY before running this notebook'
print('Setup OK. Python', sys.version)

## 1. The Tools

Before looking at the agents, let's test the individual tools.
These are pure functions — no LLM involved — so they're fast and deterministic.

In [ ]:
from tools.medical_search import search_medical_info

# test medical search
result = search_medical_info('chest pain radiating to jaw differential diagnosis')
print(result[:800])

In [ ]:
from tools.drug_checker import check_drug_interactions

# classic dangerous interaction
result = check_drug_interactions(
    new_drug='warfarin',
    current_medications=['aspirin', 'metformin', 'lisinopril']
)
print(result)

In [ ]:
from tools.calculator import calculate_bmi, assess_vitals, calculate_dosage

bmi = calculate_bmi(weight_kg=95, height_cm=175)
print('BMI:', json.dumps(bmi, indent=2))

vitals = assess_vitals(heart_rate=115, systolic_bp=160, diastolic_bp=95, temperature_c=38.5)
print('\nVitals assessment:')
print(vitals)

dosage = calculate_dosage(drug='amoxicillin', patient_age=35, condition='community acquired pneumonia')
print('\nDosage reference:')
print(dosage)

## 2. Individual Agent Test

Now let's test the SymptomAnalyzer in isolation before running the full pipeline.
This is the pattern I used when debugging — test nodes individually first.

In [ ]:
from agents.symptom_analyzer import SymptomAnalyzer

analyzer = SymptomAnalyzer()

test_state = {
    'age': 45,
    'gender': 'female',
    'symptoms': ['chest pain', 'shortness of breath', 'palpitations'],
    'symptom_description': 'Sharp chest pain for 2 days, worsens on deep breathing. Heart racing occasionally.',
    'medical_history': ['anxiety disorder'],
    'current_medications': ['sertraline'],
    'allergies': []
}

result = analyzer.run(test_state)
print('Severity:', result['severity_level'])
print('\nAnalysis preview:')
print(result['symptom_analysis'][:600])

## 3. Full Pipeline — Standard Case

Let's run the complete graph for a medium-severity case.
We'll use `stream()` to see each node's output as it completes.

In [ ]:
import uuid
from agents.orchestrator import compiled_graph

# 45-year-old with palpitations -- TC007-style case
patient_id = f'notebook_{uuid.uuid4().hex[:6]}'

initial_state = {
    'patient_id': patient_id,
    'age': 45,
    'gender': 'female',
    'symptoms': ['palpitations', 'dizziness', 'chest fluttering', 'fatigue'],
    'symptom_description': (
        'Episodes of racing heart, sudden onset, last 10-20 minutes. '
        'Light-headed during episodes. No syncope. Gets worse with caffeine. '
        'Had 4 episodes in the past month.'
    ),
    'medical_history': ['hypothyroidism', 'anxiety disorder'],
    'current_medications': ['levothyroxine 100mcg', 'sertraline 50mg'],
    'allergies': ['codeine'],
    'symptom_analysis': None,
    'severity_level': None,
    'diagnosis': None,
    'differential_diagnoses': None,
    'treatment_plan': None,
    'drug_interactions': None,
    'final_report': None,
    'messages': [],
    'error': None,
    'iteration_count': 0,
}

config = {'configurable': {'thread_id': patient_id}}
final_state = None

print('Running pipeline...\n')
for chunk in compiled_graph.stream(initial_state, config=config, stream_mode='values'):
    if chunk.get('symptom_analysis') and 'severity_level' in chunk:
        print(f'[Symptom Analyzer] Done. Severity: {chunk["severity_level"]}')
    if chunk.get('diagnosis'):
        diffs = chunk.get('differential_diagnoses', [])
        print(f'[Diagnosis Agent] Done. Differentials: {diffs[:3]}')
    if chunk.get('treatment_plan'):
        print('[Treatment Agent] Done.')
    if chunk.get('final_report'):
        print('[Report Generator] Done.')
    final_state = chunk

print('\nPipeline complete.')

In [ ]:
# display the final report
from IPython.display import Markdown, display

display(Markdown(final_state['final_report']))

In [ ]:
# show drug interactions found
print('Drug Interactions:')
print(final_state.get('drug_interactions', 'None checked'))

## 4. Emergency Routing

Now let's test the emergency pathway — the graph should route directly to
`emergency_report` and skip the full diagnosis/treatment pipeline.

In [ ]:
emergency_id = f'emergency_{uuid.uuid4().hex[:6]}'

emergency_state = {
    'patient_id': emergency_id,
    'age': 67,
    'gender': 'male',
    'symptoms': ['sudden vision loss right eye', 'facial drooping', 'slurred speech', 'arm weakness'],
    'symptom_description': (
        'All symptoms started 15 minutes ago, sudden onset. '
        'Right side face drooping, speech slurred, right arm weak, right eye vision gone. '
        'History of hypertension and previous TIA.'
    ),
    'medical_history': ['hypertension', 'atrial fibrillation', 'TIA (2 years ago)'],
    'current_medications': ['warfarin', 'amlodipine'],
    'allergies': [],
    'symptom_analysis': None,
    'severity_level': None,
    'diagnosis': None,
    'differential_diagnoses': None,
    'treatment_plan': None,
    'drug_interactions': None,
    'final_report': None,
    'messages': [],
    'error': None,
    'iteration_count': 0,
}

e_config = {'configurable': {'thread_id': emergency_id}}
e_final = None

print('Running emergency case...\n')
nodes_visited = []
for chunk in compiled_graph.stream(emergency_state, config=e_config, stream_mode='values'):
    if chunk.get('severity_level') and chunk['severity_level'] not in nodes_visited:
        nodes_visited.append(chunk['severity_level'])
        print(f'Severity: {chunk["severity_level"]}')
    if chunk.get('final_report') and e_final is None:
        print('Emergency report generated.')
    e_final = chunk

print(f'\nDiagnosis ran: {e_final.get("diagnosis") is not None}')  # should be False
print(f'Treatment ran: {e_final.get("treatment_plan") is not None}')  # should be False
print(f'Emergency report generated: {e_final.get("final_report") is not None}')  # should be True
print('\nSeverity:', e_final.get('severity_level'))

In [ ]:
display(Markdown(e_final['final_report']))

## 5. Memory — Multi-turn Conversation

The LangGraph checkpointer (MemorySaver) allows follow-up turns in the same
conversation by using the same `thread_id`.

This would let a patient come back and say "the symptoms got worse" and
the system could see the prior context. The infrastructure is set up;
the UI just doesn't use it yet.

In [ ]:
from memory.patient_memory import load_patient_history, save_visit, get_visit_summary

# save the completed visit
save_visit(patient_id, final_state)
print('Visit saved.')

# load it back
history = load_patient_history(patient_id)
print(f'Visits on record: {len(history["visits"])}')

summary = get_visit_summary(patient_id)
print('\n' + summary)

## 6. Running the Evaluation

Run a single test case from the evaluation suite.
Full eval takes ~3-4 minutes and costs API credits.

In [ ]:
from evaluation.eval_framework import load_test_cases, run_evaluation

# run just the back pain case (TC008) — fast, low complexity
result = run_evaluation(case_id='TC008', save_results=False)

if result.get('cases'):
    case = result['cases'][0]
    if 'scores' in case:
        scores = case['scores']
        print(f"Relevance: {scores.get('relevance', '?'):.2f}")
        print(f"Completeness: {scores.get('completeness', '?'):.2f}")
        print(f"Safety: {scores.get('safety_flags', '?')}")
        print(f"Hallucination risk: {scores.get('hallucination_risk', '?')}")
        print(f"Severity correct: {scores.get('severity_correct', '?')}")
        print(f"\nReasoning: {scores.get('reasoning', '')}")

In [ ]:
# Uncomment to run the full eval suite (~3-4 minutes, costs API credits)
# full_results = run_evaluation(save_results=True)
# print(json.dumps(full_results.get('aggregate', {}), indent=2))